# LSTM for RUL Prediction (sketch)


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# Load and preprocess data (as above)
data = pd.read_csv('../data/raw/train_FD001.txt', sep='\s+', header=None)
cols = ['engine_no', 'cycle'] + [f'op_setting_{i}' for i in range(1,4)] + [f'sensor_{i}' for i in range(1,22)]
data.columns = cols
max_cycles = data.groupby('engine_no')['cycle'].transform('max')
data['RUL'] = max_cycles - data['cycle']

# For simplicity, use only the last 30 cycles of each engine for LSTM
def windowed_data(df, window_size=30):
    X, y = [], []
    for eid in df['engine_no'].unique():
        e = df[df['engine_no'] == eid]
        for i in range(len(e)-window_size):
            X.append(e.iloc[i:i+window_size][[f'sensor_{i}' for i in range(1,22)]].values)
            y.append(e.iloc[i+window_size]['RUL'])
    return np.array(X), np.array(y)

X, y = windowed_data(data)

# PyTorch Dataset/Loader
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = RULDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# LSTM Model
class SimpleLSTM(nn.Module):
    def __init__(self, input_dim=21, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.out = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.out(hn[-1]).squeeze()

model = SimpleLSTM()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop (demo - only 3 epochs, expand as you like)
for epoch in range(3):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f'Epoch {epoch+1} | Loss: {loss.item():.4f}')